# Naive Bayes
Bayesian statistics applied as a classification technique truly emerged in the 1960s to solve text information retrieval, though the underlying theory and early applications date back to the mid-1700s (Reverend Thomas Bayes). 

Naive Bayes is a generative statistical inference technique. Like other generative methods, it tries to estimate the Bayes optimal classifier by factoring the posterior into the likelihood and prior (see our repository's [README.md](../../../../README.md) and the classification [README.md](../../../README.md) for more). However, Naive Bayes takes this a step further by assuming its features are *class-conditionally independent*. This allows us to break down the Bayes optimal rule as follows:

$$
f(\vec{x}) = \text{argmax}_c (P(Y=y_c|\vec{x})) \\
P(Y=y_c|\vec{x}) = \frac{P(X=\vec{x}|y_c) \cdot P(y_c)}{P(\vec{x})} \\
P(X=\vec{x}|y_c) = P(x_1, x_2, ..., x_p | y_c) = \prod_{i=1}^p P(x_i|y_c)
$$

Here, we simply have to capture the independent class-conditional distributions for each feature individually and multiply them to estimate the joint distribution. In the real world, this is a heavy and often unrealistic bias, but it is computationally incredibly efficient.

Like our other generative methods, we can estimate the class prior $P(y_c)$ by its proportion in the training dataset. Since the marginal distribution of the sample $P(\vec{x})$ is a constant denominator across all classes, we can ignore it. Finally, by applying a log transformation (which is rank-preserving), we turn the multiplications into additions to prevent floating-point underflow and maintain computational integrity. This gives us our final, equivalent class score to optimize over:

$$
f(\vec{x}) = \text{argmax}_c \left( \log(P(y_c)) + \sum_{i=1}^p \log(P(x_i | y_c)) \right)
$$

In practice, we simply look at the data type of each individual feature to determine which likelihood distribution to apply. 

Because of our naive independence assumption, we don't have to calculate massive, multi-dimensional distributions. Instead:
*   For a **continuous feature**, we can assume it follows a single-variable (univariate) Gaussian distribution and compute its individual probability density $\text{pdf}_i(x_i)$.
*   For a **binary/boolean feature** (e.g., word presence), we use a simple Bernoulli probability $P(x_i | y_c)$.
*   For **count/frequency features** (e.g., word counts), we use a Multinomial probability.

Ultimately, we compute the log-likelihood for each feature independently, and simply sum them together along with the log of the class prior. This generates the final class score that we maximize to make our prediction.

For training we estimate the class specific distributions for each feature, then use those for inference.

(tk) rewrite bernoulli to make sure you get it
(Tk), dont rehash LDA but it. belongs here
(Tk), write up your classification function example here, simple, then for others, finally way to combine with example and then classic two goals then two datasets.

### Guassian Features
(tk)

### Bernoulli Features
The Bernoulli distribution models binary outcomes (features that are strictly 0 or 1). It uses a probability $p$ to represent the chance that the outcome is 1 (by convention), meaning the alternative (outcome 0) is mathematically forced to be $1 - p$.

We can express this switch mathematically for a single feature $x_i$ using exponents:
$$
P(x_i | y_c) = p_{ci}^{x_i} (1 - p_{ci})^{(1 - x_i)}
$$

If we are modeling this in our class conditional likelihood across all features, we simply apply a rank perserving log transformation we to turn the multiplications into a computationally stable sum:

$$
f(\vec{x}) = \text{argmax}_c \left( \log(P(y_c)) + \sum_{i=1}^p \left( x_i \log(p_{ci}) + (1 - x_i) \log(1 - p_{ci}) \right) \right)
$$

### Multinomial Features
The multinomial distribution assigns a probability to a vector of event counts (integers), $P(\vec{x})$, using a probability mass function based on the outcome probabilities of single events ($\vec{p}$). It does this using combinatorics:

$$
P(\vec{x}) = \frac{N!}{\prod_{i=1}^p x_i!} \cdot \prod_{i=1}^p p_i^{x_i}
$$

This scales the probability of seeing that exact quantity of events by the number of combinatorial ways to arrange those events.

Because this combinatorial weight $\left( \frac{N!}{\prod_{i=1}^p x_i!} \right)$ depends only on the new document and is constant across all classes, it can be factored out and ignored when maximizing the class conditional likelihood. After applying a log transformation (which is rank-preserving/monotonic) for computational stability, we get our equivalent function to maximize:

$$
f(\vec{x}) = \text{argmax}_c (P(\vec{x} | y_c) \cdot P(y_c)) \\
= \text{argmax}_c \left( \frac{N!}{\prod_{i=1}^p x_i!} \cdot \prod_{i=1}^p p_{ci}^{x_i} \cdot P(y_c) \right) \\
= \text{argmax}_c \left( \sum_{i=1}^p x_i \log(p_{ci}) + \log(P(y_c)) \right)
$$

To get the learned parameter $\vec{p}_c$ during training:

$$
p_{ic} = \frac{N_{ic}}{N_c}
$$

Here, you treat all of a class's samples as one giant pooled event, and calculate the proportion of each feature's frequency to get its probability. 

When you treat a set of features this way, you are assuming that their specific order does not matter (e.g., the "bag-of-words" model), as implicit in the multinomial distribution.

### Bernoulli Features
Bernoulli distribution models binary outcomes with a probability that its 1 (by convention), the alternative is always $1 - p$.

So if we were modeling this in our class conditional likelihood:

$$
f(x) = \text{argmax}_c (p_c(x) \cdot p_c)
$$